# 01. LLaMAC EDA: Polars-first inspection notebook

이 노트북은 `data/processed/dataset_index.csv`와 `data/extracted/`를 기준으로
다운로드/압축해제가 제대로 되었는지 확인하고, 라벨/자극/센서 파일 구조를 빠르게 탐색합니다.

원칙:
- 표/집계는 Polars eager/lazy API를 사용합니다.
- plotting 경계에서만 작은 집계 결과를 NumPy/list로 변환합니다.
- 전체 biosignal CSV 16,200개를 모두 읽지 않고, 빠른 샘플 기반 sanity check를 먼저 수행합니다.
- heatmap은 participant-file completeness, questionnaire correlation, Valence-Arousal, intended-vs-reported 라벨 확인에 사용합니다.


## 0. Runtime configuration and imports

첫 셀은 경로와 EDA 샘플 크기만 설정합니다. `SIGNAL_SAMPLE_FILES`를 키우면 biosignal 샘플 집계가 더 오래 걸립니다.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find repo root from either JupyterLab cwd or notebook-directory cwd."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "llamac_research").exists():
            return candidate
    raise RuntimeError(f"Could not locate llamac_research repo root from {start}")


ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
EXTRACTED_DIR = ROOT / "data" / "extracted"
PROCESSED_DIR = ROOT / "data" / "processed"
INDEX_PATH = PROCESSED_DIR / "dataset_index.csv"

SIGNAL_SAMPLE_FILES = 40
PLOT_DPI = 120

plt.style.use("seaborn-v0_8-whitegrid")
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

print(ROOT)
assert INDEX_PATH.exists(), f"Missing {INDEX_PATH}; run the downloader/prepare step first."


## 1. Plotting and label helpers

아래 helper들은 모두 작은 집계 결과를 시각화하기 위한 용도입니다. 원본 테이블 처리는 Polars로 유지합니다.


In [ ]:
EMOTION_ID_TO_LABEL = {
    1: "neutral",
    2: "fun",
    3: "sadness",
    4: "anger",
    5: "fear",
}
EMOTION_IDS = sorted(EMOTION_ID_TO_LABEL)
EMOTION_LABELS = [EMOTION_ID_TO_LABEL[i] for i in EMOTION_IDS]
LABEL_COLS = ["Valence", "Arousal", "Dominance", "Liking", "EmotType", "EmotStr", "Seen"]


def file_family_expr() -> pl.Expr:
    """Classify LLaMAC extracted files into answer/band/eeg/respiration families."""
    return (
        pl.when(pl.col("file_name") == "answer.csv")
        .then(pl.lit("answer"))
        .when(pl.col("file_name").str.starts_with("band_"))
        .then(pl.lit("band"))
        .when(pl.col("file_name").str.starts_with("eeg_"))
        .then(pl.lit("eeg"))
        .when(pl.col("file_name").str.starts_with("respiration_"))
        .then(pl.lit("respiration"))
        .otherwise(pl.lit("other"))
        .alias("family")
    )


def add_target_columns(frame: pl.DataFrame) -> pl.DataFrame:
    """Add intended and self-reported emotion labels for EDA cross-tabs."""
    intended = (
        pl.when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(1, 10))
        .then(1)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(11, 20))
        .then(2)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(21, 30))
        .then(3)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(31, 40))
        .then(4)
        .when(pl.col("NoVideo").cast(pl.Int64, strict=False).is_between(41, 50))
        .then(5)
        .otherwise(None)
        .cast(pl.Int64)
        .alias("IntendedType")
    )
    return frame.with_columns(
        intended,
        pl.col("EmotType").cast(pl.Int64, strict=False).alias("ReportedType"),
    )


def emotion_name_expr(column: str, alias: str | None = None) -> pl.Expr:
    """Map numeric emotion ids to readable labels inside a Polars expression."""
    expr = pl.col(column)
    mapped = pl.when(expr == 1).then(pl.lit("neutral"))
    mapped = mapped.when(expr == 2).then(pl.lit("fun"))
    mapped = mapped.when(expr == 3).then(pl.lit("sadness"))
    mapped = mapped.when(expr == 4).then(pl.lit("anger"))
    mapped = mapped.when(expr == 5).then(pl.lit("fear")).otherwise(pl.lit("unknown"))
    return mapped.alias(alias or f"{column}_label")


def dense_count_matrix(frame: pl.DataFrame, row_col: str, col_col: str, value_col: str = "count") -> tuple[np.ndarray, list, list]:
    """Convert a small Polars count table into a dense matrix for heatmaps."""
    rows = frame.select(row_col).unique().sort(row_col).to_series().to_list()
    cols = frame.select(col_col).unique().sort(col_col).to_series().to_list()
    row_idx = {value: i for i, value in enumerate(rows)}
    col_idx = {value: j for j, value in enumerate(cols)}
    matrix = np.zeros((len(rows), len(cols)), dtype=float)
    for row_value, col_value, count in frame.select([row_col, col_col, value_col]).iter_rows():
        matrix[row_idx[row_value], col_idx[col_value]] = float(count)
    return matrix, rows, cols


def plot_heatmap(
    matrix: np.ndarray,
    x_labels: Iterable,
    y_labels: Iterable,
    *,
    title: str,
    xlabel: str = "",
    ylabel: str = "",
    cmap: str = "viridis",
    annotate: bool = False,
    fmt: str = ".0f",
    figsize: tuple[float, float] = (8, 5),
    cbar_label: str = "count",
) -> None:
    """Matplotlib heatmap helper with optional cell annotations."""
    x_labels = list(x_labels)
    y_labels = list(y_labels)
    fig, ax = plt.subplots(figsize=figsize, dpi=PLOT_DPI)
    im = ax.imshow(matrix, aspect="auto", cmap=cmap)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(np.arange(len(x_labels)), labels=[str(v) for v in x_labels], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(y_labels)), labels=[str(v) for v in y_labels])
    if annotate and matrix.size <= 400:
        threshold = np.nanmax(matrix) / 2 if matrix.size else 0
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                value = matrix[i, j]
                color = "white" if value > threshold else "black"
                ax.text(j, i, format(value, fmt), ha="center", va="center", color=color, fontsize=8)
    fig.tight_layout()
    plt.show()


## 2. Load the prepared dataset index with Polars

`dataset_index.csv` is the lightweight manifest created during data preparation. This cell uses `scan_csv()` first, then materializes the small index table.


In [ ]:
index_lf = pl.scan_csv(INDEX_PATH)
index = index_lf.with_columns(file_family_expr()).collect()

print(f"index rows: {index.height:,}")
print(f"participants: {index['participant_id'].n_unique():,}")
display(index.head(10))


## 3. File-family overview

This checks whether each expected file family is present and how much disk volume each family occupies.


In [ ]:
family_summary = (
    index.group_by("family")
    .agg(
        pl.len().alias("files"),
        (pl.col("size_bytes").sum() / 1024**2).round(2).alias("size_mib"),
        pl.col("file_name").n_unique().alias("unique_file_names"),
    )
    .sort("family")
)
display(family_summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=PLOT_DPI)
families = family_summary["family"].to_list()
axes[0].bar(families, family_summary["files"].to_numpy())
axes[0].set_title("File count by family")
axes[0].set_ylabel("files")
axes[1].bar(families, family_summary["size_mib"].to_numpy(), color="tab:orange")
axes[1].set_title("Disk size by family")
axes[1].set_ylabel("MiB")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 4. Participant completeness heatmap

Rows are participants and columns are file families. A healthy extraction should show one `answer.csv` and 50 trials each for `band`, `eeg`, and `respiration` per participant.


In [ ]:
participant_family_counts = (
    index.group_by(["participant_id", "family"])
    .agg(pl.len().alias("count"))
    .sort(["participant_id", "family"])
)
completeness_matrix, participant_labels, family_labels = dense_count_matrix(
    participant_family_counts, "participant_id", "family"
)

expected = {"answer": 1, "band": 50, "eeg": 50, "respiration": 50}
completeness_wide = participant_family_counts.pivot(
    index="participant_id", on="family", values="count", aggregate_function="first"
).fill_null(0).sort("participant_id")
missing_or_extra = completeness_wide.filter(
    (pl.col("answer") != expected["answer"])
    | (pl.col("band") != expected["band"])
    | (pl.col("eeg") != expected["eeg"])
    | (pl.col("respiration") != expected["respiration"])
)
print(f"participants with unexpected family counts: {missing_or_extra.height}")
display(missing_or_extra)

plot_heatmap(
    completeness_matrix,
    family_labels,
    participant_labels,
    title="Extracted file counts per participant",
    xlabel="file family",
    ylabel="participant_id",
    annotate=False,
    figsize=(7, 14),
    cbar_label="files",
)


## 5. Load all `answer.csv` tables with Polars

The answer tables are small, so they are concatenated eagerly after adding `SubjectID` from the folder/index.


In [ ]:
answer_tables: list[pl.DataFrame] = []
for participant_id, relative_path in index.filter(pl.col("file_name") == "answer.csv").select(
    ["participant_id", "relative_path"]
).iter_rows():
    answer_tables.append(
        pl.read_csv(EXTRACTED_DIR / relative_path).with_columns(
            pl.lit(int(participant_id)).alias("SubjectID")
        )
    )

answers = pl.concat(answer_tables, how="diagonal_relaxed").select(
    ["SubjectID", "Trial", "NoVideo", "Valence", "Arousal", "Dominance", "Liking", "EmotType", "EmotStr", "Seen"]
)
answers = add_target_columns(answers)

print(f"answer rows: {answers.height:,}")
print(f"subjects: {answers['SubjectID'].n_unique():,}")
print(f"trials per subject: {answers.group_by('SubjectID').len()['len'].min()}..{answers.group_by('SubjectID').len()['len'].max()}")
display(answers.head(10))


## 6. Questionnaire missingness and descriptive statistics

This cell confirms whether the main label/questionnaire columns are complete and gives compact descriptive statistics.


In [ ]:
missingness = answers.select([pl.col(col).is_null().sum().alias(col) for col in LABEL_COLS])
display(Markdown("### Missing values"))
display(missingness)

display(Markdown("### Descriptive statistics"))
display(answers.select(LABEL_COLS).describe())


## 7. Label distributions

All counts are computed with Polars; plotting uses the resulting small count tables.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6), dpi=PLOT_DPI)
axes = axes.ravel()
for ax, col in zip(axes, LABEL_COLS):
    counts = answers.group_by(col).agg(pl.len().alias("count")).sort(col)
    ax.bar([str(v) for v in counts[col].to_list()], counts["count"].to_numpy())
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("trials")
axes[-1].axis("off")
fig.suptitle("Questionnaire / reported-label distributions", y=1.02)
fig.tight_layout()
plt.show()

emotion_counts = (
    answers.with_columns(emotion_name_expr("ReportedType", "reported_label"))
    .group_by(["ReportedType", "reported_label"])
    .agg(pl.len().alias("count"))
    .sort("ReportedType")
)
display(emotion_counts)


## 8. Questionnaire correlation heatmap

This is useful for quickly seeing how Valence/Arousal/Dominance/Liking and reported emotion scales co-vary.


In [ ]:
corr = answers.select(LABEL_COLS).corr()
corr_matrix = corr.to_numpy()
plot_heatmap(
    corr_matrix,
    LABEL_COLS,
    LABEL_COLS,
    title="Questionnaire Pearson correlation heatmap",
    cmap="coolwarm",
    annotate=True,
    fmt=".2f",
    figsize=(7, 6),
    cbar_label="correlation",
)
display(corr)


## 9. Valence × Arousal heatmap

This checks the affective rating space directly. Dense cells indicate common self-report combinations.


In [ ]:
valence_arousal_counts = answers.group_by(["Valence", "Arousal"]).agg(pl.len().alias("count"))
va_matrix, valence_labels, arousal_labels = dense_count_matrix(valence_arousal_counts, "Valence", "Arousal")
plot_heatmap(
    va_matrix,
    arousal_labels,
    valence_labels,
    title="Valence × Arousal trial counts",
    xlabel="Arousal",
    ylabel="Valence",
    cmap="magma",
    annotate=True,
    figsize=(6, 5),
    cbar_label="trials",
)
display(valence_arousal_counts.sort(["Valence", "Arousal"]))


## 10. Intended stimulus emotion vs reported emotion

Rows are intended emotion derived from `NoVideo`; columns are self-reported `EmotType`. This heatmap highlights label disagreement and possible confounding.


In [ ]:
intended_reported_counts = (
    answers.with_columns(
        emotion_name_expr("IntendedType", "intended_label"),
        emotion_name_expr("ReportedType", "reported_label"),
    )
    .group_by(["intended_label", "reported_label"])
    .agg(pl.len().alias("count"))
)
# Keep semantic emotion order rather than alphabetical order.
ordered_counts = intended_reported_counts.with_columns(
    pl.col("intended_label").replace_strict(EMOTION_LABELS, EMOTION_IDS).alias("intended_order"),
    pl.col("reported_label").replace_strict(EMOTION_LABELS, EMOTION_IDS).alias("reported_order"),
).sort(["intended_order", "reported_order"])

matrix = np.zeros((len(EMOTION_LABELS), len(EMOTION_LABELS)), dtype=float)
for intended, reported, count in ordered_counts.select(["intended_label", "reported_label", "count"]).iter_rows():
    matrix[EMOTION_LABELS.index(intended), EMOTION_LABELS.index(reported)] = count

plot_heatmap(
    matrix,
    EMOTION_LABELS,
    EMOTION_LABELS,
    title="Intended stimulus emotion × reported emotion",
    xlabel="reported EmotType",
    ylabel="intended from NoVideo",
    cmap="YlGnBu",
    annotate=True,
    figsize=(7, 5),
    cbar_label="trials",
)
display(ordered_counts)


## 11. Familiarity / seen status by reported emotion

`Seen` can be a confounder. This cell checks whether familiarity is evenly distributed across reported emotion labels.


In [ ]:
seen_reported_counts = (
    answers.with_columns(emotion_name_expr("ReportedType", "reported_label"))
    .group_by(["Seen", "reported_label"])
    .agg(pl.len().alias("count"))
    .with_columns(
        pl.col("reported_label").replace_strict(EMOTION_LABELS, EMOTION_IDS).alias("reported_order")
    )
    .sort(["Seen", "reported_order"])
)
seen_matrix = np.zeros((len(seen_reported_counts["Seen"].unique().sort()), len(EMOTION_LABELS)), dtype=float)
seen_values = seen_reported_counts["Seen"].unique().sort().to_list()
for seen, reported, count in seen_reported_counts.select(["Seen", "reported_label", "count"]).iter_rows():
    seen_matrix[seen_values.index(seen), EMOTION_LABELS.index(reported)] = count

plot_heatmap(
    seen_matrix,
    EMOTION_LABELS,
    seen_values,
    title="Seen × reported emotion counts",
    xlabel="reported emotion",
    ylabel="Seen",
    cmap="PuBuGn",
    annotate=True,
    figsize=(7, 3.5),
    cbar_label="trials",
)
display(seen_reported_counts)


## 12. Inspect one band sensor file

The band files contain GSR, PPG, and skin temperature streams with separate timestamps. This is a single-file schema and timing sanity check.


In [ ]:
sample_band_row = index.filter(pl.col("family") == "band").sort(["participant_id", "file_name"]).row(0, named=True)
sample_band_path = EXTRACTED_DIR / sample_band_row["relative_path"]
signal = pl.read_csv(sample_band_path)

print(f"participant={sample_band_row['participant_id']}, file={sample_band_row['file_name']}")
print(f"rows={signal.height:,}, columns={signal.columns}")
display(signal.head(8))

timing_summary = signal.select(
    pl.len().alias("rows"),
    (pl.col("GSR_Time").max() - pl.col("GSR_Time").min()).alias("gsr_duration_s"),
    (pl.col("PPG_Time").max() - pl.col("PPG_Time").min()).alias("ppg_duration_s"),
    (pl.col("SKT_Time").max() - pl.col("SKT_Time").min()).alias("skt_duration_s"),
)
display(timing_summary)


## 13. Raw waveform sanity plot

This plots one trial's GSR, PPG, and SKT traces. It is intentionally a direct visual sanity check before feature engineering.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=False, dpi=PLOT_DPI)
for ax, value_col, time_col in zip(axes, ["GSR", "PPG", "SKT"], ["GSR_Time", "PPG_Time", "SKT_Time"], strict=True):
    time_zeroed = signal[time_col].to_numpy() - signal[time_col].to_numpy()[0]
    ax.plot(time_zeroed, signal[value_col].to_numpy(), linewidth=1)
    ax.set_title(value_col)
    ax.set_xlabel("seconds from trial start")
    ax.set_ylabel(value_col)
fig.tight_layout()
plt.show()


## 14. Sampled band-file summary with Polars

This reads only the first `SIGNAL_SAMPLE_FILES` band files, producing a quick sensor summary without scanning the full biosignal corpus.


In [ ]:
sample_band_files = index.filter(pl.col("family") == "band").sort(["participant_id", "file_name"]).head(SIGNAL_SAMPLE_FILES)
summary_rows: list[dict[str, float | int | str]] = []
for row in sample_band_files.iter_rows(named=True):
    frame = pl.read_csv(EXTRACTED_DIR / row["relative_path"])
    stats = frame.select(
        pl.len().alias("rows"),
        (pl.col("PPG_Time").max() - pl.col("PPG_Time").min()).alias("duration_s"),
        pl.col("GSR").mean().alias("GSR_mean"),
        pl.col("GSR").std().alias("GSR_std"),
        pl.col("PPG").mean().alias("PPG_mean"),
        pl.col("PPG").std().alias("PPG_std"),
        pl.col("SKT").mean().alias("SKT_mean"),
        pl.col("SKT").std().alias("SKT_std"),
    ).row(0, named=True)
    summary_rows.append({
        "participant_id": int(row["participant_id"]),
        "file_name": row["file_name"],
        **stats,
    })

signal_summary = pl.DataFrame(summary_rows)
display(signal_summary.head(12))
display(signal_summary.select(["rows", "duration_s", "GSR_mean", "PPG_mean", "SKT_mean"]).describe())


## 15. Sampled signal-summary visualizations

These plots help catch obvious extraction or sensor-scale issues before model feature extraction.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=PLOT_DPI)
axes[0].hist(signal_summary["duration_s"].to_numpy(), bins=12, color="tab:blue")
axes[0].set_title("Sampled band-file duration")
axes[0].set_xlabel("seconds")
axes[0].set_ylabel("files")

axes[1].scatter(signal_summary["PPG_mean"].to_numpy(), signal_summary["PPG_std"].to_numpy(), alpha=0.8)
axes[1].set_title("PPG mean vs std")
axes[1].set_xlabel("PPG mean")
axes[1].set_ylabel("PPG std")

axes[2].scatter(signal_summary["GSR_mean"].to_numpy(), signal_summary["SKT_mean"].to_numpy(), alpha=0.8, color="tab:green")
axes[2].set_title("GSR mean vs SKT mean")
axes[2].set_xlabel("GSR mean")
axes[2].set_ylabel("SKT mean")
fig.tight_layout()
plt.show()
